# 🚽 Sprint 5 — Corona Review Intelligence Chatbot

**RAG-powered chatbot** for analyzing Corona toilet product reviews and technical documents.

| Component | Technology |
|---|---|
| LLM | GPT-4o-mini |
| Vector Store | ChromaDB |
| Embeddings | text-embedding-3-small |
| Framework | LangChain + FastAPI |
| Tracing | LangSmith |
| Context | CSVs + PDFs + Word Docs |

> **What makes this special:** Maya auto-detects if you're a business analyst or a customer and adapts her answers accordingly. Plus a Product Liability Radar that separates real product defects from service issues.

## 1. Setup & Imports

In [1]:
import os
import json
import csv
from pathlib import Path
from collections import Counter
from datetime import datetime

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.memory import ConversationBufferMemory
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langsmith import Client
from langsmith.run_helpers import traceable
from langsmith.wrappers import wrap_openai
from openai import OpenAI

# Set working directory to project root
os.chdir('/Users/ompandya/Desktop/chatbot-corona')
load_dotenv()

# Wrap OpenAI so ALL calls are traced in LangSmith automatically
raw_client = OpenAI()
client = wrap_openai(raw_client)

print('✅ Setup complete.')
print(f'📁 Working directory: {os.getcwd()}')

✅ Setup complete.
📁 Working directory: /Users/ompandya/Desktop/chatbot-corona


## 2. What is RAG?

**Retrieval Augmented Generation (RAG)** solves a fundamental problem: LLMs don't know your private data.

```
User Question
     │
     ▼
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  Embed Question │────▶│ Search ChromaDB  │────▶│ Top-K Chunks    │
│  (vectors)      │     │ (similarity)     │     │ (most relevant) │
└─────────────────┘     └──────────────────┘     └────────┬────────┘
                                                           │
                                                           ▼
                                                  ┌─────────────────┐
                                                  │  GPT-4o-mini    │
                                                  │  (generates     │
                                                  │   answer)       │
                                                  └─────────────────┘
```

This means Maya answers from Corona's **actual review data and product docs**, not general knowledge.

## 3. Loading the Data

In [2]:
# Load the main review dataset
df = pd.read_csv('./context/Reviews_Reason_Classification.csv')
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)

print(f'📊 Total reviews: {len(df)}')
print(f'🛍️  Unique products: {df["Product Name"].nunique()}')
print(f'🌍 Marketplaces: {df["MarketPlace"].unique().tolist()}')
print(f'📅 Date range: {df["Date"].min().date()} → {df["Date"].max().date()}')
print()
print('Sentiment distribution:')
print(df['Sentiment_GPT'].value_counts())
df.head(3)

📊 Total reviews: 221
🛍️  Unique products: 49
🌍 Marketplaces: ['HomeCenter', 'Corona']
📅 Date range: 2015-09-18 → 2024-04-29

Sentiment distribution:
Sentiment_GPT
Positive    160
Negative     37
Neutral      23
Name: count, dtype: int64


,SKU,Product Name,Category,Author,Stars,Subject,Reviews,Review_English,Date,MarketPlace,Sentiment_GPT,YearMonth,Negative_Reason,Positive_Reason,Neutral_Reason
0,101651001,Sanitario Power One Blanco,Toilets,María Félix,5,Muy cómodo,"Cómodo, ahorrador de agua y bonito diseño. Bue...","Comfortable, water-saving, and has a nice desi...",2022-05-01,HomeCenter,Positive,2022-05,NaN,Toilet,NaN
1,101651001,Sanitario Power One Blanco,Toilets,Jorge Rhenals,5,Espectacular producto.,Compré hace unas semanas el producto y me fasc...,I bought the product a few weeks ago and I lov...,2022-07-16,HomeCenter,Positive,2022-07,NaN,General,NaN
2,101651001,Sanitario Power One Blanco,Toilets,NaN,5,Excelente,"Es un sanitario fuerte, muy econòmico en cuant...","It's a strong toilet, very economical in terms...",2022-11-12,HomeCenter,Positive,2022-11,NaN,General,NaN


In [3]:
# Document loaders — CSV, PDF, DOCX
from pypdf import PdfReader
from docx import Document as DocxDocument

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

def load_csv_as_documents(filepath):
    df_local = pd.read_csv(filepath)
    docs = []
    for _, row in df_local.iterrows():
        parts = []
        for field, label in [
            ('Product Name','Product'),('SKU','SKU'),('Stars','Rating'),
            ('MarketPlace','Marketplace'),('Sentiment_GPT','Sentiment'),
            ('Review_English','Review'),('Negative_Reason','Negative Reason'),
            ('Positive_Reason','Positive Reason'),
        ]:
            if field in row and pd.notna(row[field]):
                parts.append(f'{label}: {row[field]}')
        content = '\n'.join(parts)
        if content.strip():
            docs.append(Document(
                page_content=content,
                metadata={
                    'type': 'review',
                    'source': os.path.basename(filepath),
                    'product': str(row.get('Product Name', '')),
                    'sentiment': str(row.get('Sentiment_GPT', '')),
                    'marketplace': str(row.get('MarketPlace', '')),
                }
            ))
    return docs

def load_pdf(filepath):
    reader = PdfReader(filepath)
    text = ''.join([page.extract_text() or '' for page in reader.pages])
    chunks = splitter.split_text(text)
    return [Document(page_content=c, metadata={'type': 'product_doc', 'source': os.path.basename(filepath)}) for c in chunks]

def load_docx(filepath):
    doc = DocxDocument(filepath)
    text = '\n'.join([p.text for p in doc.paragraphs if p.text.strip()])
    chunks = splitter.split_text(text)
    return [Document(page_content=c, metadata={'type': 'product_doc', 'source': os.path.basename(filepath)}) for c in chunks]

print('✅ Loaders ready.')

✅ Loaders ready.


In [4]:
# Load all documents from context folder
all_docs = []
context_dir = './context'
file_summary = {'csv': 0, 'pdf': 0, 'docx': 0}

for filename in sorted(os.listdir(context_dir)):
    filepath = os.path.join(context_dir, filename)
    ext = filename.lower().split('.')[-1]
    if ext == 'csv':
        docs = load_csv_as_documents(filepath)
        file_summary['csv'] += len(docs)
    elif ext == 'pdf':
        docs = load_pdf(filepath)
        file_summary['pdf'] += len(docs)
    elif ext == 'docx':
        docs = load_docx(filepath)
        file_summary['docx'] += len(docs)
    else:
        continue
    all_docs.extend(docs)
    print(f'  ✅ {filename}: {len(docs)} chunks')

print(f'\n📦 Total chunks indexed: {len(all_docs)}')
print(f'   From CSVs:  {file_summary["csv"]}')
print(f'   From PDFs:  {file_summary["pdf"]}')
print(f'   From DOCXs: {file_summary["docx"]}')

  ✅ 121611001-SANITARIO-NYREN-BCO-ficha-tecnica-comercial.pdf: 0 chunks
  ✅ 121611001-SANITARIO-NYREN-BLANCO-instructivo-instalacion.pdf: 1 chunks
  ✅ 278471001-FT-SAC-SANITARIO-ALUVIA-RD.pdf: 5 chunks
  ✅ Negative_Reviews_Classified.csv: 37 chunks
  ✅ Reviews_English.csv: 221 chunks
  ✅ Reviews_GPT4o_Sentiment.csv: 221 chunks
  ✅ Reviews_Reason_Classification.csv: 221 chunks
  ✅ Sanitario_Aluvia_Plus.docx: 27 chunks
  ✅ Sanitario_Cascade.docx: 23 chunks
  ✅ Sanitario_Cima.docx: 18 chunks
  ✅ Sanitario_Smart.docx: 23 chunks

📦 Total chunks indexed: 797
   From CSVs:  700
   From PDFs:  6
   From DOCXs: 91


In [5]:
# Embed and store in ChromaDB
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    persist_directory='./chroma_db_notebook'
)
print(f'✅ Vector store built with {len(all_docs)} chunks!')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Vector store built with 797 chunks!


## 4. ChromaDB Inspection

Verify what's actually stored in the vector database.

In [6]:
import chromadb

chroma_client = chromadb.PersistentClient(path='./chroma_db_notebook')
collection = chroma_client.get_collection('langchain')

print(f'📦 Total chunks stored in ChromaDB: {collection.count()}')

# Sample a few records
sample = collection.get(include=['documents', 'metadatas'], limit=3)
for i, (text, meta) in enumerate(zip(sample['documents'], sample['metadatas'])):
    print(f'\n--- Record {i+1} ---')
    print(f'📌 Source: {meta.get("source", "unknown")}')
    print(f'🏷️  Type: {meta.get("type", "unknown")}')
    print(f'🔹 Preview: {text[:200]}...')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


📦 Total chunks stored in ChromaDB: 2391

--- Record 1 ---
📌 Source: unknown
🏷️  Type: review
🔹 Preview: Product: Sanitario San Giorgio Alongado Blanco
SKU: 121361001
Rating: 5
Marketplace: HomeCenter
Sentiment: Neutral
Review: Review: N/a...

--- Record 2 ---
📌 Source: unknown
🏷️  Type: review
🔹 Preview: Product: Sanitario Montecarlo Alongado Blanco
SKU: O29161001
Rating: 5
Marketplace: Corona
Sentiment: Positive
Review: Optimal attention.
Positive Reason: General...

--- Record 3 ---
📌 Source: unknown
🏷️  Type: review
🔹 Preview: Product: Sanitario San Giorgio Alongado Blanco
SKU: 121361001
Rating: 5
Marketplace: HomeCenter
Sentiment: Positive
Review: Great product, easy to install! I only needed to buy a flexible hose and a f...


## 5. RAG Chatbot with Memory

In [7]:
CHAT_MODEL = 'gpt-4o-mini'
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=False)

ANALYST_SIGNALS = [
    'which product', 'compare', 'most complaints', 'worst', 'best performing',
    'fix first', 'priority', 'defect', 'sentiment', 'marketplace',
    'improve', 'report', 'analysis', 'trend', 'insight', 'negative reviews'
]

def detect_mode(question):
    """Auto-detect if user is analyst (business) or customer."""
    if any(kw in question.lower() for kw in ANALYST_SIGNALS):
        return 'analyst'
    return 'customer'

@traceable(project='corona-toilet-reviews')
def ask(question):
    """RAG pipeline: retrieve → prompt → generate → remember."""
    # Step 1: Retrieve relevant chunks
    docs = vectorstore.similarity_search(question, k=6)
    context = '\n\n'.join([doc.page_content for doc in docs])
    sources = list(set([doc.metadata.get('source','') for doc in docs]))
    
    # Step 2: Get conversation history (memory)
    history = memory.load_memory_variables({})
    chat_history = history.get('chat_history', '')
    
    # Step 3: Detect mode and build prompt
    mode = detect_mode(question)
    
    if mode == 'analyst':
        persona = """You are Maya, Corona's sharpest product intelligence analyst.
Lead with the most important insight. Be punchy. Use emojis as markers.
Max 4-5 bullet points. End with a 💡 recommendation."""
    else:
        persona = """You are Maya, a friendly Corona product assistant.
Be warm and helpful. Give clear simple answers. Use emojis 😊."""
    
    prompt = f"""{persona}

Context from documents:
{context}

Chat History:
{chat_history}

Question: {question}
Answer:"""

    # Step 4: Call GPT (wrapped with LangSmith tracing)
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{'role': 'user', 'content': prompt}]
    )
    answer = response.choices[0].message.content
    
    # Step 5: Save to memory
    memory.save_context({'input': question}, {'output': answer})
    
    return answer, sources, mode

print('✅ Chatbot ready! Mode detection: analyst / customer')

✅ Chatbot ready! Mode detection: analyst / customer


In [8]:
# Test 1 — Analyst question
print('🔍 Question: Which products have the most complaints?')
print(f'   Mode detected: {detect_mode("Which products have the most complaints?")}')
print()
answer1, sources1, mode1 = ask('Which products have the most complaints?')
print(answer1)

🔍 Question: Which products have the most complaints?
   Mode detected: analyst



Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


🔍 **Key Insight: Product Failures in Focus!**  
- ⚠️ **Sanitario Ultra Ahorrador Powermax Blanco** (SKU: O20801001) holds a dismal rating of **1**, signaling a severe trust deficit.  
- 🚽 **3 out of 6** purchased toilets have reportedly broken, signaling a troubling quality control issue.  
- 📞 Customers face frustrating warranty service experiences, with incorrect installation claims and delayed resolutions from support.  
- 🙍‍♀️ The sentiment is overwhelmingly **negative**, indicating that these issues could tarnish brand reputation further.

💡 **Recommendation:** Immediate quality audits and proactive customer service improvement initiatives must be implemented to restore consumer trust and enhance product reliability. Consider a product recall or replacement for existing faulty units!


In [9]:
# Test 2 — Memory test (follow-up without repeating context)
print('🔍 Follow-up: What specifically are they complaining about?')
answer2, _, _ = ask('What specifically are they complaining about?')
print(answer2)

🔍 Follow-up: What specifically are they complaining about?
Customers are mainly complaining about two issues: 

1. **Product Color**: Many requested a **white toilet**, but received a **beige toilet** instead, leading to frustration. 😟

2. **Unclear Product Images**: Some customers found the product photos unclear, making it difficult to understand what they were purchasing. 📷❓

These issues have created a lot of dissatisfaction. If you need help with anything else, feel free to ask! 😊


In [10]:
# Test 3 — Customer question (from product docs)
print('🔍 Customer question: How do I install the Sanitario Nyren?')
print(f'   Mode detected: {detect_mode("How do I install the Sanitario Nyren?")}')
print()
answer3, sources3, mode3 = ask('How do I install the Sanitario Nyren?')
print(answer3)
print(f'\n📄 Sources used: {sources3}')

🔍 Customer question: How do I install the Sanitario Nyren?
   Mode detected: customer

To install the **Sanitario San Giorgio Alongado Blanco**, follow these simple steps:

1. **Prepare the Area**: Make sure the space where you'll be installing the toilet is clean and clear of any obstacles. 🧹

2. **Check Water Supply**: Ensure that the water supply line is properly directed where the toilet will be installed.

3. **Position the Wax Ring**: Place a wax ring on the toilet flange, which is the part that attaches to the floor. This helps create a seal when the toilet is placed on it. 🛠️

4. **Place the Toilet**: Carefully lift the toilet and position it over the wax ring and flange. Align the holes in the toilet base with the flange bolts. 

5. **Secure the Toilet**: Gently press down to compress the wax ring and secure the toilet to the floor by tightening the nuts onto the flange bolts, but be careful not to overtighten! 🔩

6. **Connect the Water Supply**: Attach the water supply line t

## 6. LangSmith Tracing

Every call to `ask()` is automatically traced in LangSmith via the `@traceable` decorator and `wrap_openai`.

This means we can monitor:
- Input/output of every call
- Latency per request
- Token usage and cost
- User feedback (👍/👎)

In [11]:
# Verify LangSmith connection
try:
    ls_client = Client()
    projects = list(ls_client.list_projects())
    print(f'✅ LangSmith connected!')
    print(f'   Projects: {[p.name for p in projects]}')
except Exception as e:
    print(f'❌ LangSmith error: {e}')

print(f'\nTracing enabled: {os.getenv("LANGCHAIN_TRACING_V2", "not set")}')
print(f'Project: {os.getenv("LANGCHAIN_PROJECT", "not set")}')

✅ LangSmith connected!
   Projects: ['corona-toilet-reviews']

Tracing enabled: true
Project: corona-toilet-reviews


## 7. Feedback Logging

In [12]:
FEEDBACK_CSV = './feedback_log.csv'

def log_feedback(question, answer, rating, sources):
    """Log user feedback to CSV. Rating: 👍 or 👎"""
    file_exists = os.path.exists(FEEDBACK_CSV)
    with open(FEEDBACK_CSV, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['timestamp', 'question', 'answer', 'rating', 'sources'])
        writer.writerow([
            datetime.now().isoformat(timespec='seconds'),
            question, answer[:300], rating,
            '; '.join(sources)
        ])
    print(f'✅ Feedback logged: {rating}')

# Example — log the last answer
log_feedback(
    'Which products have the most complaints?',
    answer1,
    '👍',
    sources1
)

# Show feedback log
if os.path.exists(FEEDBACK_CSV):
    fb_df = pd.read_csv(FEEDBACK_CSV)
    print(f'\nFeedback log ({len(fb_df)} entries):')
    fb_df.tail(3)

✅ Feedback logged: 👍

Feedback log (1 entries):


## 8. Product Liability Radar

The key business insight: **not all negative reviews are equal.**
- 🔴 **Product defects** → Corona engineering team needs to fix the product
- 📦 **Service issues** → HomeCenter/logistics needs to fix the process

The radar separates them automatically.

In [13]:
DEFECT_KEYWORDS = [
    'design flaw', 'manufacturing', 'defective', 'broken out of box',
    'cracks', 'leaks', 'clogged', 'poor quality', 'installation impossible',
    'separated from wall', 'bad smell', 'holes', 'valve broken', 'tank cracks',
    'spare parts', 'replacement parts', 'wrong color', 'damaged'
]
SERVICE_KEYWORDS = [
    'delivery', 'never arrived', 'warranty', 'customer service',
    'no response', 'waiting', 'refund', 'sent wrong', 'installation service'
]

def run_radar():
    results = vectorstore.similarity_search(
        'defective broken poor quality design flaw clogged leaks', k=30
    )
    product_defects, service_issues = [], []
    for doc in results:
        text = doc.page_content.lower()
        product = doc.metadata.get('product', 'Unknown')
        if any(kw in text for kw in DEFECT_KEYWORDS):
            product_defects.append(product)
        elif any(kw in text for kw in SERVICE_KEYWORDS):
            service_issues.append(product)
    return product_defects, service_issues

defects, services = run_radar()
print(f'🚨 Product defects found: {len(defects)}')
print(f'📦 Service issues found:  {len(services)}')
print(f'\nTop defective products:')
for product, count in Counter(defects).most_common(5):
    print(f'  {product}: {count} mentions')

🚨 Product defects found: 24
📦 Service issues found:  3

Top defective products:
  : 21 mentions
  Sanitario Montecarlo Redondo Bone: 3 mentions


## 9. FastAPI Web App

The chatbot runs as a full web application using FastAPI + Uvicorn.

In [14]:
%pip install -q fastapi uvicorn
print('✅ FastAPI + Uvicorn ready.')
print()
print('To start the web app:')
print('  python app.py')
print('Then open: http://127.0.0.1:8000')
print()
print('API Endpoints:')
print('  POST /chat     — send a question, get an answer')
print('  POST /feedback — send thumbs up/down')
print('  GET  /radar    — run product liability scan')
print('  GET  /surprise — get a surprise insight')
print('  POST /clear    — clear conversation memory')

Note: you may need to restart the kernel to use updated packages.
✅ FastAPI + Uvicorn ready.

To start the web app:
  python app.py
Then open: http://127.0.0.1:8000

API Endpoints:
  POST /chat     — send a question, get an answer
  POST /feedback — send thumbs up/down
  GET  /radar    — run product liability scan
  GET  /surprise — get a surprise insight
  POST /clear    — clear conversation memory


## 10. Data Visualizations

All charts use real Corona review data. Each one tells a different part of the story.

In [15]:
COLORS = {'Positive': '#2ecc71', 'Neutral': '#f1c40f', 'Negative': '#e74c3c'}
TEMPLATE = 'plotly_white'

# Chart 1 — Sentiment Distribution Donut
sentiment_counts = df['Sentiment_GPT'].value_counts().reset_index()
sentiment_counts.columns = ['Sentiment', 'Count']

fig = px.pie(
    sentiment_counts, names='Sentiment', values='Count',
    title='<b>Overall Sentiment Distribution</b>',
    color='Sentiment', color_discrete_map=COLORS,
    hole=0.45, template=TEMPLATE
)
fig.update_traces(
    textposition='outside', textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Share: %{percent}<extra></extra>',
    pull=[0.05, 0, 0]
)
fig.update_layout(
    title_x=0.5, title_font_size=16,
    width=550, height=450,
    margin=dict(t=80, b=60, l=40, r=40),
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5)
)
fig.show()

In [16]:
# Chart 2 — Sentiment by Marketplace (Grouped Bar)
mp = df.groupby(['MarketPlace', 'Sentiment_GPT']).size().reset_index(name='Count')

fig = px.bar(
    mp, x='MarketPlace', y='Count', color='Sentiment_GPT',
    title='<b>Sentiment by Marketplace</b>',
    color_discrete_map=COLORS, barmode='group',
    text='Count', template=TEMPLATE
)
fig.update_traces(textposition='outside', textfont_size=11)
fig.update_layout(
    title_x=0.5, title_font_size=16,
    width=600, height=450,
    margin=dict(t=80, b=60, l=40, r=40),
    legend_title='Sentiment',
    xaxis_title='Marketplace', yaxis_title='Number of Reviews'
)
fig.show()

In [17]:
# Chart 3 — Product Defects vs Service Issues (Fixed)
radar_df = pd.DataFrame({
    'Category': ['Product Defects', 'Service Issues'],
    'Count': [len(defects), len(services)],
})

fig = px.bar(
    radar_df, x='Category', y='Count',
    title='<b>🚨 Product Defects vs Service Issues</b>',
    color='Category',
    color_discrete_map={'Product Defects': '#e74c3c', 'Service Issues': '#f39c12'},
    text='Count', template=TEMPLATE
)
fig.update_traces(
    textposition='outside', textfont_size=16,
    width=0.4
)
fig.update_layout(
    title_x=0.5, title_font_size=16,
    width=550, height=480,
    margin=dict(t=80, b=60, l=60, r=60),
    showlegend=False,
    yaxis=dict(range=[0, max(len(defects), len(services)) * 1.3]),
    xaxis_title='', yaxis_title='Count'
)
fig.show()

In [18]:
# Chart 4 — Top 10 Products by Negative Reviews (Horizontal Bar)
neg_df = df[df['Sentiment_GPT'] == 'Negative']
top_neg = neg_df['Product Name'].value_counts().head(10).reset_index()
top_neg.columns = ['Product', 'Count']
top_neg['Product'] = top_neg['Product'].str.replace('Sanitario ', '')
top_neg = top_neg.sort_values('Count')

fig = px.bar(
    top_neg, x='Count', y='Product', orientation='h',
    title='<b>Top 10 Products by Negative Reviews</b>',
    color='Count', color_continuous_scale='OrRd',
    text='Count', template=TEMPLATE
)
fig.update_traces(textposition='outside', textfont_size=11)
fig.update_layout(
    title_x=0.5, title_font_size=16,
    width=700, height=500,
    margin=dict(t=80, b=60, l=180, r=80),
    coloraxis_showscale=False,
    xaxis_title='Number of Negative Reviews', yaxis_title=''
)
fig.show()

In [19]:
# Chart 5 — Why Are Customers Unhappy? (Negative Reason Breakdown)
neg_reasons = df[df['Sentiment_GPT']=='Negative']['Negative_Reason'].value_counts().reset_index()
neg_reasons.columns = ['Reason','Count']
neg_reasons = neg_reasons[neg_reasons['Reason'].notna()]

fig = px.pie(
    neg_reasons, names='Reason', values='Count',
    title='<b>Root Cause of Negative Reviews</b>',
    color_discrete_sequence=['#e74c3c', '#e67e22', '#c0392b'],
    hole=0.45, template=TEMPLATE
)
fig.update_traces(
    textposition='outside', textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>%{percent}<extra></extra>'
)
fig.update_layout(
    title_x=0.5, title_font_size=16,
    width=550, height=450,
    margin=dict(t=80, b=60, l=40, r=40),
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5)
)
fig.show()

In [20]:
# Chart 6 — Monthly Sentiment Trend (Line Chart)
monthly = df[df['Sentiment_GPT'].isin(['Positive','Negative','Neutral'])]
monthly = monthly.groupby(['YearMonth','Sentiment_GPT']).size().reset_index(name='Count')
monthly = monthly[monthly['YearMonth'] != 'NaT']

fig = px.line(
    monthly, x='YearMonth', y='Count', color='Sentiment_GPT',
    title='<b>Sentiment Trend Over Time</b>',
    color_discrete_map=COLORS,
    markers=True, template=TEMPLATE
)
fig.update_traces(line_width=2.5, marker_size=6)
fig.update_layout(
    title_x=0.5, title_font_size=16,
    width=800, height=450,
    margin=dict(t=80, b=80, l=60, r=40),
    xaxis_title='Month', yaxis_title='Number of Reviews',
    xaxis_tickangle=-45,
    legend_title='Sentiment'
)
fig.show()

In [21]:
# Chart 7 — SUNBURST: Sentiment broken down by Product (unique & beautiful)
# Shows the full picture in one chart — which products drive which sentiments
top_products = df['Product Name'].value_counts().head(10).index.tolist()
sun_df = df[df['Product Name'].isin(top_products)].copy()
sun_df['Product Short'] = sun_df['Product Name'].str.replace('Sanitario ', '')
sun_counts = sun_df.groupby(['Sentiment_GPT', 'Product Short']).size().reset_index(name='Count')

fig = px.sunburst(
    sun_counts, path=['Sentiment_GPT', 'Product Short'], values='Count',
    title='<b>Sentiment by Product — Sunburst View</b>',
    color='Sentiment_GPT',
    color_discrete_map=COLORS,
    template=TEMPLATE
)
fig.update_traces(
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Share: %{percentParent}<extra></extra>',
    insidetextorientation='radial'
)
fig.update_layout(
    title_x=0.5, title_font_size=16,
    width=650, height=600,
    margin=dict(t=80, b=40, l=40, r=40)
)
fig.show()

In [22]:
# Chart 8 — Star Rating Distribution with Average Line
star_counts = df['Stars'].value_counts().sort_index().reset_index()
star_counts.columns = ['Stars', 'Count']
avg_stars = df['Stars'].mean()

fig = px.bar(
    star_counts, x='Stars', y='Count',
    title=f'<b>Star Rating Distribution</b> (avg: {avg_stars:.1f}⭐)',
    color='Stars',
    color_continuous_scale=['#e74c3c','#e67e22','#f1c40f','#a8d8a8','#2ecc71'],
    text='Count', template=TEMPLATE
)
fig.add_vline(
    x=avg_stars, line_dash='dash', line_color='#2c3e50', line_width=2,
    annotation_text=f'Average: {avg_stars:.1f}',
    annotation_position='top right',
    annotation_font_size=12
)
fig.update_traces(textposition='outside', textfont_size=11)
fig.update_layout(
    title_x=0.5, title_font_size=16,
    width=600, height=450,
    margin=dict(t=80, b=60, l=60, r=60),
    coloraxis_showscale=False,
    xaxis_title='Star Rating', yaxis_title='Number of Reviews',
    xaxis=dict(tickmode='linear', tick0=0, dtick=1)
)
fig.show()

In [23]:
# Chart 9 — Most At-Risk Products from Radar
defect_counter = Counter(defects)
top_defects = defect_counter.most_common(8)
products_d = [x[0].replace('Sanitario ','') for x in top_defects]
counts_d = [x[1] for x in top_defects]

fig = go.Figure(go.Bar(
    x=counts_d, y=products_d,
    orientation='h',
    text=counts_d, textposition='outside',
    marker=dict(
        color=counts_d,
        colorscale='Reds',
        showscale=False
    ),
    hovertemplate='<b>%{y}</b><br>Defect mentions: %{x}<extra></extra>'
))
fig.update_layout(
    title=dict(text='<b>🚨 Most At-Risk Products (Product Defects Only)</b>', x=0.5, font_size=16),
    template=TEMPLATE,
    width=700, height=500,
    margin=dict(t=80, b=60, l=180, r=80),
    xaxis_title='Defect Mentions', yaxis_title='',
    xaxis=dict(range=[0, max(counts_d)*1.3])
)
fig.show()

## 11. Summary

| Feature | Status |
|---|---|
| RAG with ChromaDB | ✅ |
| GPT-4o-mini | ✅ |
| Conversation Memory | ✅ |
| PDF + DOCX + CSV ingestion | ✅ |
| Auto mode detection (analyst vs customer) | ✅ |
| Product Liability Radar | ✅ |
| `@traceable` + `wrap_openai` | ✅ |
| LangSmith Tracing | ✅ |
| Feedback logging to CSV | ✅ |
| FastAPI Web App | ✅ |
| 🤯 Surprise Me feature | ✅ |
| Dark mode UI | ✅ |
| 9 Plotly charts | ✅ |